In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1432_Sonia_Vihar_Delhi_DPCC_1Day.csv")

In [4]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,175.49,273.60,6.48,15.90,22.37,32.83,3.00,1.19,12.50,...,NaN,12.41,79.70,0.49,47.77,0.0,0.0,57.27,986.95,NaN
1,2024-01-02,165.38,274.13,8.10,15.05,23.15,29.81,4.95,1.23,12.57,...,NaN,12.23,76.43,0.44,66.53,0.0,0.0,79.44,986.11,NaN
2,2024-01-03,185.92,278.94,16.05,14.50,30.55,35.98,2.36,1.60,8.07,...,NaN,11.31,89.99,0.38,112.76,0.0,0.0,38.06,985.99,NaN
3,2024-01-04,224.70,355.33,17.82,26.85,35.53,39.02,4.49,1.48,3.98,...,NaN,11.96,87.19,0.29,161.49,0.0,0.0,27.46,986.28,NaN
4,2024-01-05,146.26,256.80,8.67,33.39,24.69,43.41,5.04,1.49,10.10,...,NaN,12.74,90.68,0.30,98.30,0.0,0.0,15.27,986.20,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,129.36,143.68,32.94,47.45,52.03,42.84,2.27,1.32,10.61,...,NaN,15.13,86.16,1.27,115.34,0.0,0.0,9.68,994.00,NaN
362,2024-12-28,68.76,75.73,82.48,59.48,98.69,30.54,1.07,1.27,3.00,...,NaN,15.43,88.60,0.45,187.72,0.0,0.0,18.62,994.00,NaN
363,2024-12-29,73.08,93.50,74.07,59.49,91.86,29.01,1.20,1.12,2.24,...,NaN,15.03,83.46,0.89,239.65,0.0,0.0,83.76,993.74,NaN
364,2024-12-30,70.38,100.29,38.12,46.52,55.74,30.40,3.55,1.27,12.87,...,NaN,13.67,78.71,0.78,248.58,0.0,0.0,74.74,994.21,NaN


In [5]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [6]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp        0
PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
NOx (ppb)        0
NH3 (µg/m³)      0
SO2 (µg/m³)      0
CO (mg/m³)       0
Ozone (µg/m³)    0
AT (°C)          0
RH (%)           0
WS (m/s)         0
WD (deg)         0
RF (mm)          0
TOT-RF (mm)      0
SR (W/mt2)       0
BP (mmHg)        0
dtype: int64


In [7]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [8]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 18)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         175.49        273.60        6.48        15.90   
1  2024-01-02         165.38        274.13        8.10        15.05   
2  2024-01-03         185.92        278.94       16.05        14.50   
3  2024-01-04         224.70        355.33       17.82        26.85   
4  2024-01-05         146.26        256.80        8.67        33.39   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  AT (°C)  \
0      22.37        32.83         3.00        1.19          12.50    12.41   
1      23.15        29.81         4.95        1.23          12.57    12.23   
2      30.55        35.98         2.36        1.60           8.07    11.31   
3      35.53        39.02         4.49        1.48           3.98    11.96   
4      24.69        43.41         5.04        1.49          10.10    12.74   

   RH (%)  WS (m/s)  WD (deg)  RF (mm)  TOT-RF (mm)  SR (W/mt2)  BP (mmHg)  
0   

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [10]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.226720,0.588703,-0.639395,-1.628131,-0.587571,0.350410,-1.127008,-0.203958,-1.299529,-1.747075,1.130815,-0.150619,-1.655578,0.0,0.0,-1.671379,0.719487
1,2024-01-02,1.069233,0.593787,-0.529036,-1.694282,-0.542999,-0.030779,-0.736982,-0.126118,-1.296176,-1.769437,0.894828,-0.358488,-1.349568,0.0,0.0,-1.224476,0.596299
2,2024-01-03,1.389192,0.639928,0.012543,-1.737086,-0.120136,0.748009,-1.255017,0.593902,-1.511697,-1.883734,1.873414,-0.607930,-0.595475,0.0,0.0,-2.058615,0.578700
3,2024-01-04,1.993281,1.372718,0.133121,-0.775955,0.164439,1.131723,-0.828988,0.360382,-1.707582,-1.802981,1.671346,-0.982093,0.199399,0.0,0.0,-2.272290,0.621230
4,2024-01-05,0.771395,0.427545,-0.490205,-0.266984,-0.454997,1.685836,-0.718981,0.379842,-1.414473,-1.706077,1.923209,-0.940520,-0.831343,0.0,0.0,-2.518016,0.609497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.508138,-0.657588,1.163143,0.827226,1.107309,1.613889,-1.273018,0.049022,-1.390048,-1.409153,1.597014,3.092129,-0.553390,0.0,0.0,-2.630699,1.753391
362,2024-12-28,-0.435849,-1.309415,-0.307635,1.763452,-0.241852,0.061362,-1.513035,-0.048278,-1.754517,-1.371883,1.773102,-0.316914,0.627257,0.0,0.0,-2.450487,1.753391
363,2024-12-29,-0.368555,-1.138952,-0.307635,1.764230,-0.241852,-0.131757,-1.487033,-0.340178,-1.790916,-1.421577,1.402163,1.512328,1.474328,0.0,0.0,-1.137394,1.715261
364,2024-12-30,-0.410613,-1.073817,1.516021,0.754849,1.319312,0.043691,-1.017001,-0.048278,-1.281808,-1.590538,1.059369,1.055018,1.619992,0.0,0.0,-1.319219,1.784188


In [11]:
df.to_excel("soniavihar2024.xlsx",index=False)